# Job Salary Estimator — DAY 4: Deep Learning and LLMs

Neural network on BoW features, then frontier LLMs for zero-shot salary prediction.

In [ ]:
import os
from dotenv import load_dotenv
from job_salary.evaluator import evaluate
from litellm import completion
from job_salary.items import Job
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
# Load data
try:
    train, val, test = Job.from_hub('ed-donner/job_salaries_processed')
except Exception:
    from datasets import load_dataset
    from job_salary.parser import parse
    ds = load_dataset('lukebarousse/data_jobs', split='train', trust_remote_code=True)
    items = [parse(dp) for dp in tqdm(ds.select(range(10000)))]
    items = [j for j in items if j is not None]
    n = len(items)
    train, val, test = items[:int(0.8*n)], items[int(0.8*n):int(0.9*n)], items[int(0.9*n):]

def get_text(job):
    return job.summary if job.summary else job.full or ''

print(f'Test size: {len(test):,}')

In [ ]:
# PyTorch Neural Network (optional - requires torch)
try:
    from sklearn.feature_extraction.text import HashingVectorizer
    from sklearn.model_selection import train_test_split
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    docs = [get_text(j) for j in train]
    y = np.array([float(j.salary) for j in train])
    vec = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
    X = vec.fit_transform(docs)
    X_t = torch.FloatTensor(X.toarray())
    y_t = torch.FloatTensor(y).unsqueeze(1)

    class NN(nn.Module):
        def __init__(self, in_size):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(in_size, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(),
                nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1)
            )
        def forward(self, x):
            return self.layers(x)

    model = NN(X_t.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(2):
        model.train()
        for i in range(0, len(X_t), 64):
            bx, by = X_t[i:i+64], y_t[i:i+64]
            opt.zero_grad()
            loss = nn.MSELoss()(model(bx), by)
            loss.backward()
            opt.step()

    def nn_pricer(job):
        model.eval()
        with torch.no_grad():
            x = vec.transform([get_text(job)])
            x_t = torch.FloatTensor(x.toarray())
            return max(0, model(x_t).item())

    evaluate(nn_pricer, test, size=min(200, len(test)))
except Exception as e:
    print(f'Neural net skipped: {e}')

In [ ]:
# Zero-shot Frontier LLMs
def messages_for(job):
    text = get_text(job)
    msg = f"Estimate the yearly salary in USD for this job. Respond with only the number, no explanation.\n\n{text}"
    return [{"role": "user", "content": msg}]

print(test[0].salary)
print(test[0].summary or test[0].full[:500])

In [ ]:
def gpt_nano(job):
    r = completion(model='openai/gpt-4.1-nano', messages=messages_for(job))
    return r.choices[0].message.content

evaluate(gpt_nano, test, size=min(50, len(test)), workers=2)

In [ ]:
def claude_opus(job):
    r = completion(model='anthropic/claude-opus-4-5', messages=messages_for(job))
    return r.choices[0].message.content

evaluate(claude_opus, test, size=min(50, len(test)), workers=2)

In [ ]:
# Try Gemini or Groq for cost-effective inference
def groq_llama(job):
    r = completion(model='groq/llama-3.3-70b-versatile', messages=messages_for(job))
    return r.choices[0].message.content

evaluate(groq_llama, test, size=min(100, len(test)), workers=3)